# Evacuation Readiness—Load & Explore

**Challenge 4: Adaptive Evacuation Readiness & Vulnerable Community Planning**

Standard evacuation planning assumes everyone has a car, a phone, and the ability to walk out of a
building. The people who die are the ones for whom none of that is true. The useful thing about
them is that **they are findable in advance**, from public data, and that is what this notebook
builds.

You are not building a dispatch system. Nobody is being rescued in real time here. You are building
what a county emergency manager needs in **August**, so that in **September** the plan already
exists.

---

## What you'll have when this finishes

Five tables in BigQuery, in **your** project, for **one state you choose**:

| Table | What it holds |
|---|---|
| `shelters` | Every facility ever registered as a shelter in your state, with capacity, accessibility, generator, surge exposure |
| `vulnerability_tracts` | Census tracts with no-vehicle households, 65+, disability, limited English, group quarters, mobile homes |
| `power_dependent_counties` | Medicare beneficiaries relying on electricity for medical equipment, by county |
| `care_facilities` | Nursing homes with coordinates and certified bed counts |
| `hazard_tracts` | Hurricane and coastal-flood exposure per tract |

## The four lanes, and roughly what each does with the 4.5 hours

- **Data**—runs this notebook (about 40 minutes including reading it), then keeps going: the
  add-on layers in Section 10, the joins nobody pre-built, the query that answers your team's
  actual question.
- **Agent**—starts reading Section 13 now. The architecture constraint there is the single most
  likely thing to cost your team half an hour, and it is better to know at minute 5.
- **Front end**—you need a deployed endpoint before you can build against one, so the agent lane
  deploys at roughly the halfway mark. Until then: read Section 13's attribution rules, because
  they are not optional and they shape your UI.
- **Story**—you get a **short pitch deck and a quick demo**, and presentation time at this event
  is abbreviated. **Rehearse to the clock.** Start from Section 9—the numbers there are your
  opening, and they are more striking than anything you will build today.

**Read the markdown even if you skip the code.** Everything that matters about this data is in the
prose, and about half of it is a trap.

## Before you start

You will be asked to **enable APIs twice**—once when Colab Enterprise first opens, and again from
a separate button on the Colab Enterprise homepage. That second one is easy to miss. This is
expected, not an error.

This notebook also needs the Agent Platform API, because Section 3 makes one live call to Gemini.
If you have not enabled it, open **Cloud Shell** (the `>_` icon in the top right of the console) and
run:

```bash
gcloud services enable aiplatform.googleapis.com bigquery.googleapis.com
```

**If you have never used a notebook before:** each grey block below is a cell. Click it and press
**Shift+Enter** to run it. Run them in order, top to bottom. If one fails, fix it before moving on
—later cells depend on earlier ones.

## 1. Setup

One cell of configuration, and one decision: **which state**.

Evacuation is administered by states and counties, not by metro areas, so this notebook is scoped
to a state rather than a bounding box. Pick one and the whole notebook follows.

In [ ]:
# ---- the one thing you change -------------------------------------------------
STATE = "FL"          # two-letter abbreviation. See the table in Section 5.
# -------------------------------------------------------------------------------

import json, math, os, re, sys, time
import pandas as pd
import requests
from google.cloud import bigquery

# FIPS codes are fixed by federal standard and have not changed since 1970.
# Hardcoding them is safe in a way that hardcoding a table name never is.
STATE_FIPS = {
    "AL":"01","AK":"02","AZ":"04","AR":"05","CA":"06","CO":"08","CT":"09","DE":"10","DC":"11",
    "FL":"12","GA":"13","HI":"15","ID":"16","IL":"17","IN":"18","IA":"19","KS":"20","KY":"21",
    "LA":"22","ME":"23","MD":"24","MA":"25","MI":"26","MN":"27","MS":"28","MO":"29","MT":"30",
    "NE":"31","NV":"32","NH":"33","NJ":"34","NM":"35","NY":"36","NC":"37","ND":"38","OH":"39",
    "OK":"40","OR":"41","PA":"42","RI":"44","SC":"45","SD":"46","TN":"47","TX":"48","UT":"49",
    "VT":"50","VA":"51","WA":"53","WV":"54","WI":"55","WY":"56",
}
assert STATE in STATE_FIPS, f"{STATE!r} is not a state abbreviation I know. Try 'FL'."
FIPS = STATE_FIPS[STATE]

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or \
          __import__("subprocess").check_output(
              ["gcloud", "config", "get-value", "project"], text=True).strip()
DATASET = "evacuation_readiness"
LOCATION = "US"

bq = bigquery.Client(project=PROJECT)

print(f"project : {PROJECT}")
print(f"state   : {STATE}  (FIPS {FIPS})")
print(f"dataset : {PROJECT}.{DATASET}")

### Why this notebook is timed

Every step below runs inside a timer, and the appendix prints a total. That is not decoration.

You have 4.5 hours. If loading the data takes 25 minutes instead of 8, that is 17 minutes your
agent lane does not get, and the only way to know is to measure. The timings also tell *us*
whether this notebook is still fit for purpose next year, when one of these public services has
quietly become three times slower.

The same cell defines the **checks harness**. Validation runs at the end, but the list it fills up
lives here.

In [ ]:
STEPS, CHECKS = {}, []

class step:
    """Time a block of work. Never swallows an error—if a step fails you need to know."""
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time(); print(f"[{self.name}] ...", flush=True); return self
    def __exit__(self, et, ev, tb):
        STEPS[self.name] = round(time.time() - self.t0, 1)
        if et is None:
            print(f"[{self.name}] done in {STEPS[self.name]}s", flush=True)
        return False

def check(name, passed, detail=""):
    CHECKS.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": str(detail)})
    print(f"  {'PASS' if passed else 'FAIL'}  {name}  {detail}")
    return passed

def get_json(url, params=None, tries=5, timeout=120):
    """
    Fetch JSON, with backoff, and never hand a non-JSON body to .json().

    Both halves matter. FEMA's shelter service rate-limits, and it answers with HTTP 403 and an
    HTML page rather than a 429—so naive code raises a JSON parse error and you spend ten
    minutes blaming your parser instead of your request rate.
    """
    delay = 1.0
    for attempt in range(tries):
        r = requests.get(url, params=params, timeout=timeout)
        ct = r.headers.get("content-type", "")
        if r.status_code == 200 and "json" in ct:
            j = r.json()
            if isinstance(j, dict) and "error" in j:
                raise RuntimeError(f"service returned an error: {json.dumps(j['error'])[:300]}")
            return j
        # Only 429 and 5xx are worth retrying. A 400 means the request itself is wrong, and
        # asking the same wrong question five more times just delays the news by thirty seconds.
        retryable = r.status_code == 429 or r.status_code >= 500
        if not retryable or attempt == tries - 1:
            raise RuntimeError(
                f"{url} returned HTTP {r.status_code} ({ct[:40]})"
                f"{'' if not retryable else f' after {tries} attempts'}. "
                f"First 300 bytes: {r.text[:300]!r}")
        time.sleep(delay); delay *= 2

def arcgis_all(url, where, out_fields, page=2000, order="objectid", centroid=False):
    """
    Page an ArcGIS feature service and return a DataFrame.

    Note what this does NOT do: it does not use returnDistinctValues alongside paging. On FEMA's
    service that combination returns HTTP 200, a well-formed body, and an empty feature list. A
    query that succeeds and returns nothing is worse than one that fails.
    """
    if not url.rstrip("/").endswith("/query"):
        url = url.rstrip("/") + "/query"
    rows, offset = [], 0
    while offset < 500000:
        params = {"f": "json", "where": where, "outFields": out_fields,
                  "returnGeometry": "false", "resultOffset": offset,
                  "resultRecordCount": page}
        if order:
            params["orderByFields"] = order
        if centroid:
            # outSR matters: these services publish in Web Mercator, so without it you get
            # centroids in metres and every distance you compute is quietly nonsense.
            params["returnCentroid"] = "true"
            params["outSR"] = "4326"
        try:
            j = get_json(url, params)
        except RuntimeError:
            if not order:
                raise
            order = None            # not every service names its OID field 'objectid'
            continue
        feats = j.get("features", [])
        if not feats:
            break
        for f in feats:
            row = dict(f["attributes"])
            if centroid and f.get("centroid"):
                row["lon"], row["lat"] = f["centroid"].get("x"), f["centroid"].get("y")
            rows.append(row)
        offset += len(feats)
        if len(feats) < page and not j.get("exceededTransferLimit"):
            break
    return pd.DataFrame(rows)

def safe_str(s, width=None):
    """
    Cast to string WITHOUT inventing data.

    `.astype(str)` turns None into the literal string "None" and NaN into "nan". Those are
    strings. They load into BigQuery cleanly, they survive every not-null check you write,
    and they come out the other side looking like values. Zero-padding one gives you "0None".
    """
    out = s.astype("string")
    if width:
        out = out.str.strip().str.zfill(width)
    return out.where(out.notna(), None)

def load_df(df, table, schema=None):
    """Replace a table wholesale. --replace everywhere makes reruns safe."""
    ref = f"{PROJECT}.{DATASET}.{table}"
    cfg = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    if schema: cfg.schema = schema
    bq.load_table_from_dataframe(df, ref, job_config=cfg).result()
    n = bq.get_table(ref).num_rows
    print(f"  loaded {ref}: {n:,} rows")
    return n

print("helpers ready")

In [ ]:
with step("create dataset"):
    ds = bigquery.Dataset(f"{PROJECT}.{DATASET}")
    ds.location = LOCATION
    try:
        bq.create_dataset(ds)
        print(f"created {PROJECT}.{DATASET}")
    except Exception as e:
        # Tolerate "already exists" even behind a check. Checks misfire, and two teammates can
        # run this cell in the same project in the same second.
        if "Already Exists" in str(e) or "already exists" in str(e):
            print(f"{PROJECT}.{DATASET} already exists—fine, continuing")
        else:
            raise

## 2. Start with the wrong answer

Before you load anything, watch the obvious approach fail. This is the most useful ten minutes in
the notebook, and it also proves your project can reach Gemini—which you would otherwise discover
at minute 200.

**Grounding with Google Maps is your required differentiator.** It connects Gemini to live Google
Maps data: 250 million places, addresses, opening hours, whether somewhere is open *right now*.
It is genuinely excellent, and the obvious thing to do with it is ask where the shelters are.

So let's do that, and see what we get.

In [ ]:
from google import genai
from google.genai import types

os.environ["GOOGLE_CLOUD_PROJECT"]  = PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

MODEL = "gemini-3.6-flash"   # verified with Maps grounding on Vertex, 2026-08-09
# Place names for the hook prompt. Not dataset identifiers—just somewhere real to ask about.
COUNTY_HINT = {"FL": "Miami-Dade County, Florida", "TX": "Harris County, Texas",
               "GA": "Chatham County, Georgia", "NC": "New Hanover County, North Carolina",
               "LA": "Orleans Parish, Louisiana", "SC": "Charleston County, South Carolina",
               "NY": "Kings County, New York", "CA": "Los Angeles County, California"}
COUNTY = COUNTY_HINT.get(STATE, f"the largest coastal county in {STATE}")

gclient, MAPS = None, None
try:
    gclient = genai.Client(vertexai=True, project=PROJECT, location="global")
    MAPS = types.Tool(google_maps=types.GoogleMaps())
    print("Gemini client ready")
except Exception as e:
    print("Could not create the Gemini client:", e)
    print("\nOpen Cloud Shell (the >_ icon) and run:")
    print("  gcloud services enable aiplatform.googleapis.com")
    print("Then re-run this cell. The rest of the notebook works without it, but you will miss")
    print("the point of Section 2.")

def ask_maps(prompt):
    """One grounded call. Returns (text, chunks) or (None, error)."""
    r = gclient.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(tools=[MAPS]))
    gm = r.candidates[0].grounding_metadata
    chunks = [ch for ch in ((gm.grounding_chunks if gm else None) or []) if getattr(ch, "maps", None)]
    return r.text, chunks

In [ ]:
# c4_90_publish_snapshot.ipynb skips this cell
with step("hook: ask the obvious question"):
    if gclient:
        text, chunks = ask_maps(
            f"I am an emergency manager in {COUNTY}. Where could I shelter people "
            f"during a hurricane evacuation? List a few specific places with addresses.")
        print(text[:1400])
        print("\n" + "-" * 70)
        print(f"grounded in {len(chunks)} Google Maps sources, for example:")
        for ch in chunks[:5]:
            print(f"  · {ch.maps.title}  ->  {ch.maps.uri}")
    else:
        print("skipped—no Gemini client")

Read what came back before you read on.

It is fluent, it is cited, every address is real, and **it is the wrong category of building.**
Those are homeless shelters, rescue missions and family services centres—not hurricane
evacuation shelters. The model even flags it itself, unprompted: *"it's important to differentiate
these from official hurricane evacuation shelters."*

That is not a bad answer to the question. It is a correct answer to a **different** question,
because "shelter" means something different to Google Maps than it means to an emergency manager.
This is the first trap and it is a semantic one: a grounded tool will confidently answer the words
you typed rather than the thing you meant, and it will cite real sources while doing it.

Now ask the three questions an emergency manager actually has to answer.

In [ ]:
PLANNER_QUESTIONS = [
    f"For hurricane evacuation planning in {COUNTY}: which shelters can accept someone "
    f"who uses a wheelchair?",
    f"Which shelters in {COUNTY} have a backup generator, for people who depend on "
    f"electricity for medical equipment?",
    f"Which shelters in {COUNTY} are themselves inside the storm surge zone?",
]

# c4_90_publish_snapshot.ipynb skips this cell
with step("hook: ask the questions that matter"):
    if gclient:
        for q in PLANNER_QUESTIONS:
            text, chunks = ask_maps(q)
            print(f"\nQ: {q}\n")
            print(f"A: {text[:420]}")
            print(f"   [{len(chunks)} Maps sources]")
    else:
        print("skipped—no Gemini client")

### What just happened

Three questions, three variations on *"I couldn't find specific information."*

This is not a failure of the tool and it is not a prompt you can engineer your way out of. Google
Maps knows an enormous amount about **places**. It knows almost nothing about whether a place
**works for a person who cannot walk**, and nothing at all about **who lives nearby and cannot
drive**.

That gap is the entire design of this challenge:

| Question | Who can answer it |
|---|---|
| Where is it, is it open, is it still there? | **Grounding with Google Maps**, at runtime |
| Can a wheelchair get in? Is there a generator? Is it in the surge zone? | **The data you are about to load** |
| Who nearby cannot get themselves out? | **The data you are about to load** |
| Is the 2022 record still true today? | **Grounding with Google Maps**, at runtime |

Neither half is optional and neither substitutes for the other. An agent that only calls Maps
grounding gives confident, useless answers. An agent that only queries BigQuery is working from a
federal file that may be describing a building that burned down in 2023.

**Your agent has to do both, and know which to trust for what.** That is the challenge.

## 3. Who cannot evacuate themselves

Start with the sharpest signal, because it reframes everything after it.

**HHS emPOWER** publishes, every month, the number of Medicare beneficiaries in each county who
rely on **electricity-dependent medical equipment**—ventilators, oxygen concentrators, electric
beds, power wheelchairs, home dialysis. These are people for whom a power cut is not an
inconvenience.

**Licence.** This is a US Government work published by HHS ASPR and CMS. It carries a
purpose-of-use condition rather than an open licence:

> *"Use of this tool and data signifies your agreement to use it for the specified purposes and to
> make no attempt to identify any individual in this data."*

The specified purpose is emergency preparedness. That is exactly what you are doing. Do not attempt
to re-identify anyone, and do not use it for anything else.

**What it cannot do.** It is published at **county** and **ZIP** level, never census tract. So it
tells you *how many* and *roughly where*, and it will not tell you *which block*. Any team that
presents a tract-level map of power-dependent residents has invented data.

In [ ]:
EMPOWER = ("https://services2.arcgis.com/ZQ4jTQn6k7VPXEwO/arcgis/rest/services/"
           "HHS_emPOWER_REST_Service_Public/FeatureServer/2")

with step("pull emPOWER (county)"):
    # Only 3,233 counties nationally, so pull the lot and filter locally. Guessing a
    # server-side field name we have not seen is how you get an empty result and no error.
    emp_all = arcgis_all(EMPOWER, "1=1", "*")
    print(f"  national rows: {len(emp_all):,}")
    print(f"  columns: {list(emp_all.columns)[:14]} ...")

    fips_col = next((c for c in emp_all.columns if c.upper() == "FIPS_CODE"), None)
    assert fips_col, f"no FIPS column found. Columns are: {list(emp_all.columns)}"
    emp_all[fips_col] = safe_str(emp_all[fips_col], width=5)
    empower = emp_all[emp_all[fips_col].str.startswith(FIPS).fillna(False)].copy()

    # Two counts around the filter. One count of zero looks identical to "no data here".
    print(f"  {STATE} rows: {len(empower):,} of {len(emp_all):,} national")
    if len(empower) == 0:
        raise RuntimeError(
            f"No emPOWER counties matched FIPS prefix {FIPS!r}. Check the FIPS column format—"
            f"sample values: {emp_all[fips_col].head(3).tolist()}")
    display(empower.head(3))

### The number 11 is not the number 11

emPOWER suppresses small counts. From its own documentation:

> *"Any geographic area with claims totaling 1-10 is displayed as 11."*

So a county showing 11 power-dependent residents has somewhere between one and ten. It is a
**censored value**, not a measurement, and it is the single easiest way to poison an analysis in
this dataset: sum a state's counties and you inflate every rural one to 11.

Look at how many rows this affects before deciding what to do about it.

In [ ]:
with step("emPOWER: find the censored values"):
    num_cols = [c for c in empower.columns
                if empower[c].dtype.kind in "if" and c.upper() not in ("OBJECTID",)]
    key = next((c for c in num_cols if "POWER_DEPENDENT_DEVICES" in c.upper()), None)
    print(f"  power-dependent column: {key}")
    if key:
        censored = int((empower[key] == 11).sum())
        print(f"  counties reported as exactly 11 (i.e. 1-10): {censored} of {len(empower)}")
        print(f"  total across {STATE}: {int(empower[key].fillna(0).sum()):,} "
              f"(inflated by up to {censored * 10:,} because of the masking)")
        empower["power_dependent_is_censored"] = empower[key] == 11
        empower["power_dependent_dme"] = empower[key]

## 4. Pick your state—with the real numbers in front of you

These counts come from an actual run against the federal shelter file. They are here because
choosing a state blind is how a team spends an afternoon on a hurricane story in a state with one
medical-needs shelter.

**These counts cover the *usable* subset only**—records that have coordinates *and* a recorded
capacity. Section 9 reports **every** record for your state, so its numbers will be larger. For
Florida: 1,533 usable out of 2,793 total, and 88 wheelchair-recorded out of 247. Same data, two
denominators, and it is worth knowing which one you are quoting on a slide.

| State | Usable shelters | Total capacity | Wheelchair recorded YES | Generator YES | Medical-needs | Pre-landfall (EVAC) |
|---|---:|---:|---:|---:|---:|---:|
| **FL** | 1,533 | 848,784 | **88** | 109 | **151** | 1,170 |
| TX | 2,651 | 772,337 | 526 | 85 | 31 | 935 |
| GA | 1,573 | 1,138,056 | 375 | 170 | 14 | 638 |
| NC | 1,207 | 445,627 | 288 | 145 | 16 | 415 |
| LA | 1,006 | 355,010 | 125 | 52 | 14 | 529 |
| SC | 604 | 328,746 | 159 | 77 | **1** | 323 |
| NY | 3,724 | 1,639,805 | 921 | 643 | 10 | 1,708 |
| CA | 5,648 | 6,554,178 | 1,105 | 439 | 204 | 1,284 |

**Florida is the default**, and deliberately. It has the right hazard profile, the second-highest
count of medical-needs shelters in the country, and the **worst wheelchair-reporting rate of any
coastal state**—88 of 1,533. That contrast is the story.

**South Carolina is the cautionary one.** 604 shelters, 328,746 spaces, and exactly **one** with a
medical designation. If your whole demo hangs off medical-needs matching, SC will not carry it.

Any state works. States not listed will work too—you just have not seen their numbers, and
neither have we, so check Section 9 before you build a narrative on them.

## 5. Pull the shelters—once

FEMA's **ESF#6 National Shelter System** is the only standing, national, capacity-attributed
shelter inventory that exists. 71,710 records nationally.

Two things about *how* we pull it, because both are teaching points.

**One paged pull, not twenty small queries.** This service rate-limits. Around fifty small requests
in and it starts answering **HTTP 403 with an HTML page**—not a 429, no `Retry-After`, no JSON
error. A full paged pull of all 71,710 rows, by contrast, completes in about twelve seconds. It is
the *number of requests* that trips it, not the volume of data. Now imagine 150 people in this room
each firing off twenty analytical queries in the same ten minutes. **Pull once, analyse locally.**

**We name our columns, and four of them we deliberately never request.** The service exposes
`org_poc_name`, `org_poc_phone`, `org_poc_after_hours_phone` and `org_poc_email`, and 47,006 rows
have a real person's name in them. That is individual-level personal data, it is an automatic
rejection under this event's rules, and the only reliable defence is to never ask for it.

**Never `SELECT *` against this service.** That is a rule, not a preference.

In [ ]:
NSS = "https://gis.fema.gov/arcgis/rest/services/NSS/FEMA_NSS/FeatureServer/5/query"

# Explicit, and deliberately excluding every org_poc_* column. This list IS the policy.
SHELTER_FIELDS = ("shelter_id,shelter_name,address_1,city,county_parish,state,zip,fips_code,"
                  "latitude,longitude,evacuation_capacity,post_impact_capacity,ada_compliant,"
                  "wheelchair_accessible,generator_onsite,self_sufficient_electricity,"
                  "pet_accommodations_code,in_100_yr_floodplain,in_500_yr_floodplain,"
                  "in_surge_slosh_area,pre_landfall_shelter,facility_usage_code,"
                  "shelter_status_code,population_code,incident_id,match_type")

with step("pull FEMA shelters"):
    shelters = arcgis_all(NSS, f"state='{STATE}'", SHELTER_FIELDS)
    print(f"  {STATE} shelter records: {len(shelters):,}")
    if len(shelters) == 0:
        raise RuntimeError(
            f"No shelters returned for state='{STATE}'. Check the abbreviation, and check whether "
            f"gis.fema.gov is reachable from this runtime.")
    assert not any(c.startswith("org_poc") for c in shelters.columns), \
        "a personal-data column got in—stop and fix the field list"
    display(shelters.head(3))

## 6. The real problems in this data

Five of them. None raises an error. All five are the kind that quietly produce a plausible wrong
answer, which is the only kind worth teaching.

We show each one broken before fixing it. If you skip to the cleaned table you will not believe
how bad some of these are.

### Problem 1—2,205 shelters are in the Indian Ocean

Wisconsin's rows have **latitude and longitude swapped**. Not some of them: 2,205 of 2,209.
Latitude values run −92.8 to −87.1, which is Wisconsin's *longitude*; longitude runs 42.5 to 46.9,
which is its *latitude*. Only four Wisconsin rows are correct.

Every value is a well-formed float in a plausible range. Nothing raises. A **national** range check
passes, because 97% of the file is fine—which is exactly why this survived. Only a check against
the bounding box of the state a row *claims to be in* catches it.

Left alone, it relocates 2,205 shelters into open water and any spatial join against Wisconsin
returns zero rows with no explanation.

In [ ]:
with step("clean: coordinates"):
    for col in ("latitude", "longitude"):
        shelters[col] = pd.to_numeric(shelters[col], errors="coerce")

    swapped = ((shelters["state"] == "WI") &
               (shelters["latitude"] < 0) & (shelters["longitude"] > 0))
    print(f"  rows with latitude/longitude swapped: {int(swapped.sum())}")
    if swapped.any():
        shelters.loc[swapped, ["latitude", "longitude"]] = \
            shelters.loc[swapped, ["longitude", "latitude"]].values
        print("  swapped back")

    null_island = (shelters["latitude"] == 0) | (shelters["longitude"] == 0)
    print(f"  rows at exactly (0, 0)—'Null Island': {int(null_island.sum())}")

    before = len(shelters)
    shelters = shelters[
        shelters["latitude"].notna() & shelters["longitude"].notna() &
        shelters["latitude"].between(17, 72) & ~null_island].copy()
    print(f"  rows: {before:,} before, {len(shelters):,} after "
          f"({100 * len(shelters) / before:.1f}% kept)")

### Problem 2—a blank string is not a null

`ada_compliant IS NOT NULL` matches **49,356** rows nationally. That sounds like excellent
coverage. It is not: **46,338 of those rows contain a single space character.**

The true answered count is 3,080. A sixteen-fold overstatement, produced by a test that everybody
writes without thinking.

Anywhere a publisher pads with `' '`, count with `TRIM(col) NOT IN ('', 'UNK')` and never with
`IS NOT NULL`.

In [ ]:
FLAGS = ["ada_compliant", "wheelchair_accessible", "generator_onsite",
         "self_sufficient_electricity", "in_surge_slosh_area",
         "in_100_yr_floodplain", "in_500_yr_floodplain", "pre_landfall_shelter"]

with step("clean: measure real coverage"):
    rows = []
    for col in FLAGS:
        s = shelters[col].fillna("").astype(str).str.strip().str.upper()
        rows.append({"field": col,
                     "IS NOT NULL says": int(shelters[col].notna().sum()),
                     "actually answered": int(s.isin(["YES", "NO"]).sum()),
                     "YES": int((s == "YES").sum()),
                     "NO": int((s == "NO").sum()),
                     "UNK": int((s == "UNK").sum()),
                     "blank or null": int((~s.isin(["YES", "NO", "UNK"])).sum())})
    coverage = pd.DataFrame(rows)
    display(coverage)

### Problem 3—`ada_compliant` is a decoy

Look at the two accessibility columns in that table.

`ada_compliant` is the obvious name. It sits right next to the real one. And nationally it has
**63 YES rows in the entire United States**.

`wheelchair_accessible` has **14,092**.

A field that exists and looks like the right shape is not the right field. The tell here is not the
name, it is the **coverage**—so rank your candidate columns by answered-value count before you
pick one, every time.

Two more are effectively empty and should be dropped rather than trusted: `in_500_yr_floodplain`
(zero YES rows nationally) and `pre_landfall_shelter` (14).

### Problem 4—the flags are `YES`/`NO`, not `Y`/`N`—and the file is not consistent with itself

Testing `ada_compliant == 'Y'` returns zero. Not "few"—zero, in every state. Which reads exactly
like "no shelter in America is accessible," and that is a sentence somebody will put on a slide.

Meanwhile `pet_accommodations_code` in the *same table* uses whole words: `COHABIT`, `ONSITE`,
`NONE`.

One publisher, one table, several conventions. **Print `value_counts()` before you write any
filter.** A predicate that returns zero rows is indistinguishable from a feature that genuinely
does not exist, and people believe the second reading.

We convert to real booleans here, keeping `UNK` and blank as `NULL`—because "nobody recorded it"
and "no" are different facts and collapsing them destroys the most important finding in this
notebook.

In [ ]:
def yes_no_to_bool(s):
    """YES -> True, NO -> False, everything else -> None. UNK is not a no."""
    v = s.fillna("").astype(str).str.strip().str.upper()
    return v.map({"YES": True, "NO": False}).astype("boolean")

with step("clean: normalise the flags"):
    KEEP_FLAGS = ["wheelchair_accessible", "generator_onsite",
                  "self_sufficient_electricity", "in_surge_slosh_area", "in_100_yr_floodplain"]
    for col in KEEP_FLAGS:
        shelters[col] = yes_no_to_bool(shelters[col])
    # Dropped on purpose: ada_compliant (63 YES nationally), in_500_yr_floodplain (0),
    # pre_landfall_shelter (14). Keeping an empty column invites somebody to filter on it.
    shelters = shelters.drop(columns=["ada_compliant", "in_500_yr_floodplain",
                                      "pre_landfall_shelter"], errors="ignore")
    for col in ("evacuation_capacity", "post_impact_capacity"):
        shelters[col] = pd.to_numeric(shelters[col], errors="coerce")

    shelters["is_medical"] = shelters["population_code"].fillna("").str.contains("MEDICAL")
    shelters["is_functional"] = shelters["population_code"].fillna("").str.contains("FUNCTIONAL")
    shelters["usage"] = shelters["facility_usage_code"].fillna("").str.strip().replace("", None)
    print(shelters[KEEP_FLAGS].notna().sum().to_string())
    print(f"\n  medical-needs shelters in {STATE}: {int(shelters['is_medical'].sum())}")

### Problem 5—what a row actually is

`shelter_id` is unique across all 71,710 rows, so it looks like a clean facility key. It is not—
it is just the row's primary key.

The real question is whether the same *building* appears twice. Measured nationally: 71,419
distinct name-and-address combinations out of 71,710 rows. Buildings barely duplicate.

And 41,874 rows carry an `incident_id`, which looks like an activation log duplicating the
inventory. It is not: only **69** buildings appear in both sets, and **41,629** appear only in the
incident set. Those are additional distinct facilities, not repeats.

So this is a **historical registry**—every facility ever registered as a shelter, one row each,
with `incident_id` recording where the record came from. Which means the whole file is your
inventory, *and* that a building registered during a 2022 hurricane may not be a shelter today.

**That is precisely what a live Maps lookup is for**, and it is why this challenge has the
differentiator it has.

In [ ]:
with step("clean: check the grain yourself"):
    key = (shelters["shelter_name"].str.strip().str.upper() + "|" +
           shelters["address_1"].fillna("").str.strip().str.upper())
    print(f"  rows: {len(shelters):,}")
    print(f"  distinct shelter_id: {shelters['shelter_id'].nunique():,}")
    print(f"  distinct name+address: {key.nunique():,}")
    print(f"  duplication ratio: {len(shelters) / max(key.nunique(), 1):.2f}")
    has_inc = shelters["incident_id"].notna() & \
              (shelters["incident_id"].astype(str).str.strip() != "")
    print(f"  rows tagged to a past incident: {int(has_inc.sum()):,} "
          f"({100 * has_inc.mean():.0f}%)")

## 7. Load the shelters into BigQuery

From here on the data lives in your project and you query it with SQL. Everything is written with
`WRITE_TRUNCATE`, so re-running any cell is safe—the end state is the same whether this is your
first run or your fourth.

In [ ]:
with step("load shelters"):
    cols = ["shelter_id", "shelter_name", "address_1", "city", "county_parish", "state", "zip",
            "fips_code", "latitude", "longitude", "evacuation_capacity", "post_impact_capacity",
            "wheelchair_accessible", "generator_onsite", "self_sufficient_electricity",
            "in_surge_slosh_area", "in_100_yr_floodplain", "pet_accommodations_code",
            "population_code", "is_medical", "is_functional", "usage", "shelter_status_code",
            "incident_id"]
    out = shelters[[c for c in cols if c in shelters.columns]].copy()
    # fips_code is empty in this file—every Florida row is null. Dropping it rather than
    # shipping a column of "0None". See the markdown above.
    out = out.drop(columns=["fips_code"], errors="ignore")
    out["zip"] = safe_str(out["zip"].str.strip().str[:5], width=5)
    n_shelters = load_df(out, "shelters")

with step("give each shelter a real county"):
    # FEMA's own fips_code is unusable, so derive the county from the coordinates instead—
    # exact, cheap (67 polygons), and it is what lets shelters join to emPOWER.
    try:
        bq.query(f"""
        CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.shelters` AS
        SELECT s.*, c.geo_id AS county_fips, c.county_name
        FROM `{PROJECT}.{DATASET}.shelters` s
        LEFT JOIN `bigquery-public-data.geo_us_boundaries.counties` c
          ON ST_CONTAINS(c.county_geom, ST_GEOGPOINT(s.longitude, s.latitude))
        """).result()
        r = list(bq.query(f"""
            SELECT COUNT(*) n, COUNTIF(county_fips IS NOT NULL) matched,
                   COUNT(DISTINCT county_fips) counties
            FROM `{PROJECT}.{DATASET}.shelters`""").result())[0]
        print(f"  shelters placed in a county: {r.matched:,} of {r.n:,} "
              f"across {r.counties} counties")
    except Exception as e:
        print(f"  county assignment skipped: {type(e).__name__}: {str(e)[:200]}")
        print("  (the shelters table is still fine—you just have no county_fips column)")

## 8. Who lives near those shelters, and who cannot drive to one

Three sources, one table. Census tracts are the unit—roughly 4,000 people each, which is small
enough to act on and large enough that nobody is identifiable.

**American Community Survey** gives the household facts: no vehicle, 65 and over, group quarters,
mobile homes, limited English. It is in BigQuery already, so it costs you nothing to query.

Two things the ACS in BigQuery does **not** have, and you will look for both: **no disability
column** and **no living-alone column**. Neither exists in this dataset. Disability comes from CDC
below. Living-alone needs the Census API and an API key, which is why it is a bonus rather than
part of the core path—see Section 13.

Never hardcode a table name. Ask which ones exist and take the newest.

In [ ]:
with step("discover ACS tables"):
    tabs = [r.table_name for r in bq.query("""
        SELECT table_name
        FROM `bigquery-public-data.census_bureau_acs.INFORMATION_SCHEMA.TABLES`
        WHERE table_name LIKE 'censustract%'
        ORDER BY table_name DESC""").result()]
    ACS_TABLE = tabs[0]
    print(f"  found {len(tabs)} tract tables, newest is: {ACS_TABLE}")

    acs_cols = [r.column_name for r in bq.query(f"""
        SELECT column_name
        FROM `bigquery-public-data.census_bureau_acs.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name = '{ACS_TABLE}'""").result()]
    print(f"  columns: {len(acs_cols)}")

    # Build the 65+ expression from columns that actually exist, rather than typing twelve
    # names from memory and discovering at runtime that two of them do not.
    age_cols = sorted(c for c in acs_cols
                      if re.match(r"^(male|female)_(6[5-9]|7\d|8\d)", c))
    print(f"  age 65+ columns found ({len(age_cols)}): {age_cols}")
    assert age_cols, "no 65+ age columns found—inspect acs_cols before continuing"
    POP65 = " + ".join(f"IFNULL(SAFE_CAST({c} AS FLOAT64), 0)" for c in age_cols)
    for want in ("no_cars", "group_quarters", "mobile_homes",
                 "speak_spanish_at_home_low_english"):
        print(f"  {want}: {'present' if want in acs_cols else 'ABSENT'}")

### The leading zero that breaks seven states

Census tract identifiers are eleven characters and start with a two-digit state FIPS code. The
geometry table stores Arizona as `04013612800`. The ACS table stores it as `4013612800`—ten
characters, leading zero gone, because somewhere in its history that identifier passed through
something that treated it as a number.

**It does not raise.** `JOIN ... USING (geo_id)` returns zero rows, every downstream statistic comes
back null, and nothing tells you why.

Affected states are exactly those with a FIPS code below 10: Alabama, Alaska, Arizona, Arkansas,
**California**, Colorado, Connecticut. Between them that is Los Angeles, San Francisco, San Diego,
San Jose, Sacramento, Denver and Phoenix.

`LPAD(geo_id, 11, '0')` is a no-op when the zero is already there and saves you when it is not.
Free insurance—use it on every ACS join you ever write.

And note the two counts printed below. **A join that ate your data looks exactly like "there is no
data here."** One number cannot tell those apart; two can.

### Vintage: the second silent join-killer

Census tract boundaries were redrawn for the 2020 census. Florida had **4,245** tracts in 2010 and
**5,160** in 2020, and the two sets do not nest—tracts were split, merged and renumbered.

That matters here because our sources are not all the same age:

| Source | Florida tracts | Vintage |
|---|---:|---|
| ACS `censustract_2020_5yr` | 5,160 | 2020 |
| CDC SVI 2022 | 5,122 | 2020 |
| FEMA National Risk Index | 5,114 | 2020 |
| `geo_census_tracts` (the geometry table) | **4,245** | **2010** |

So the geometry table is the odd one out, and if you build your table around it—as the first
version of this notebook did—you throw away a fifth of the state before you start. We anchor on
ACS instead and attach coordinates where the older geometry can supply them.

Only **3,338** of Florida's 5,160 tracts appear in both, so anchoring on the geometry table costs
you a third of the state's mappable points.

**The fix is to stop asking the wrong service for coordinates.** FEMA's National Risk Index is
2020 vintage and its feature service will hand you a centroid per tract if you ask
(`returnCentroid=true`). One parameter matters enormously there: **`outSR=4326`**. These services
publish in Web Mercator, so without it you get centroids in metres and every distance you compute
afterwards is quietly nonsense—no error, just wrong answers with plausible magnitudes.

We take NRI's centroids first and fall back to the older geometry table for anything NRI does not
cover. In Florida that moves coordinate coverage from **65% to 100%**—NRI supplies 5,114 of the
5,160 tracts and the old geometry table covers the remaining 46.

In [ ]:
with step("pull ACS + tract geometry"):
    sql = f"""
    WITH acs AS (
      SELECT LPAD(geo_id, 11, '0') AS geo_id,
             SAFE_CAST(total_pop AS FLOAT64)  AS total_pop,
             SAFE_CAST(households AS FLOAT64) AS households,
             SAFE_CAST(no_cars AS FLOAT64)    AS households_no_vehicle,
             SAFE_CAST(group_quarters AS FLOAT64) AS group_quarters_pop,
             SAFE_CAST(mobile_homes AS FLOAT64)   AS mobile_homes,
             SAFE_CAST(speak_spanish_at_home_low_english AS FLOAT64) AS limited_english_spanish,
             SAFE_CAST(median_income AS FLOAT64)  AS median_income,
             SAFE_DIVIDE(SAFE_CAST(poverty AS FLOAT64),
                         SAFE_CAST(pop_determined_poverty_status AS FLOAT64)) AS poverty_rate,
             {POP65} AS pop_65_plus
      FROM `bigquery-public-data.census_bureau_acs.{ACS_TABLE}`
    ),
    geo AS (
      SELECT geo_id,
             SAFE_CAST(internal_point_lat AS FLOAT64) AS lat,
             SAFE_CAST(internal_point_lon AS FLOAT64) AS lon
      FROM `bigquery-public-data.geo_census_tracts.us_census_tracts_national`
      WHERE state_fips_code = '{FIPS}'
    )
    -- ACS is the spine, not the geometry. The old boundaries are a fallback for coordinates.
    SELECT a.*, g.lat AS lat_2010, g.lon AS lon_2010
    FROM acs a LEFT JOIN geo g USING (geo_id)
    WHERE a.geo_id LIKE '{FIPS}%'
    """
    tracts = bq.query(sql).to_dataframe()
    if len(tracts) == 0:
        raise RuntimeError(
            f"No ACS tracts matched the prefix {FIPS!r}. Check the LPAD—this is the "
            f"leading-zero bug, not missing data.")
    # County comes free from the identifier: the first five characters ARE the county FIPS.
    tracts["county_fips"] = tracts["geo_id"].str[:5]

NRI = ("https://services.arcgis.com/XG15cJAlne2vxtgt/arcgis/rest/services/"
       "National_Risk_Index_Census_Tracts/FeatureServer/0")

with step("tract coordinates from NRI centroids"):
    cen = arcgis_all(NRI, f"TRACTFIPS LIKE '{FIPS}%'", "TRACTFIPS",
                     order="TRACTFIPS", centroid=True)
    if "lat" not in cen.columns:
        print("  NRI did not return centroids—falling back to the 2010 geometry table alone")
        cen = pd.DataFrame(columns=["geo_id", "lat", "lon"])
    else:
        cen = cen.rename(columns={"TRACTFIPS": "geo_id"})[["geo_id", "lat", "lon"]]
        cen["geo_id"] = safe_str(cen["geo_id"], width=11)
    tracts = tracts.merge(cen, on="geo_id", how="left")

    # Prefer the 2020-vintage centroid; fall back to the 2010 internal point.
    tracts["lat"] = tracts["lat"].fillna(tracts["lat_2010"])
    tracts["lon"] = tracts["lon"].fillna(tracts["lon_2010"])
    tracts = tracts.drop(columns=["lat_2010", "lon_2010"])

    n_geo = list(bq.query(f"""
        SELECT COUNT(*) n FROM `bigquery-public-data.geo_census_tracts.us_census_tracts_national`
        WHERE state_fips_code = '{FIPS}'""").result())[0].n
    with_coords = int(tracts["lat"].notna().sum())
    print(f"  ACS tracts in {STATE}          : {len(tracts):,}   (2020 boundaries)")
    print(f"  2010 geometry table has       : {n_geo:,}   (2010 boundaries)")
    print(f"  NRI centroids returned        : {len(cen):,}")
    print(f"  tracts with coordinates       : {with_coords:,}  "
          f"({100*with_coords/max(len(tracts),1):.0f}%)")
    # Sanity-check the units. A centroid in Web Mercator is a number in the millions, and it
    # will happily flow through every downstream calculation looking like a coordinate.
    if with_coords:
        lo, hi = float(tracts["lat"].min()), float(tracts["lat"].max())
        assert 17 <= lo and hi <= 72, (
            f"latitudes out of range ({lo} to {hi})—that usually means outSR was ignored "
            f"and you have Web Mercator metres, not degrees")
        print(f"  latitude range                : {lo:.2f} to {hi:.2f}  (degrees, as intended)")

### CDC Social Vulnerability Index—and the part of it we refuse to use

The SVI gives us the two things ACS does not: **disability** and a tract-level measure of people
with **no vehicle**, plus limited English and group quarters, all in one file.

**We use Themes 1, 2 and 4, and we do not use Theme 3.**

Theme 3 is racial and ethnic minority status. This project prohibits race and ethnicity as a
**model input**, and requires them instead as a **post-hoc audit of the output**. The reasoning
matters more than the rule, and it is worth having straight before a judge asks:

- Race genuinely *does* correlate with these outcomes. Do not claim otherwise; anyone who knows the
  literature will correct you.
- But it is a **proxy**. The causal variables here are physical and structural—vehicle access,
  building type, medical dependency, distance to a shelter—and we can measure those directly.
- **Removing the column does not remove the bias.** Correlated proxies survive. This is "fairness
  through unawareness" and it does not work. The remedy is auditing what your agent recommends, not
  deleting an input.

This is consistent with Google's own published responsible-AI guidance, so it is not an ROI
invention.

Practical consequence: **`RPL_THEMES`, the overall SVI score, is contaminated**—it has Theme 3
baked in and there is no column to drop. Use `RPL_THEME1`, `RPL_THEME2` and `RPL_THEME4`
individually, and leave the composite alone.

Also: **`-999` is the missing-data sentinel.** Leave it in and your averages are spectacular.

In [ ]:
SVI = ("https://services3.arcgis.com/ZvidGQkLaDJxRSJ2/arcgis/rest/services/"
       "CDC_ATSDR_Social_Vulnerability_Index_2022_USA/FeatureServer/2")

with step("pull CDC SVI"):
    # Ask the service what fields it has rather than trusting a list we typed.
    meta = get_json(SVI, {"f": "json"})
    available = {f["name"] for f in meta.get("fields", [])}
    wanted = ["FIPS", "E_TOTPOP", "EP_DISABL", "EP_AGE65", "EP_NOVEH", "EP_LIMENG",
              "EP_MOBILE", "EP_GROUPQ", "EP_NOINT", "RPL_THEME1", "RPL_THEME2", "RPL_THEME4"]
    use = [f for f in wanted if f in available]
    missing = [f for f in wanted if f not in available]
    print(f"  requesting {len(use)} fields; not present: {missing or 'none'}")
    # RPL_THEMES is deliberately absent from `wanted`. It contains Theme 3.

    svi = arcgis_all(SVI, f"FIPS LIKE '{FIPS}%'", ",".join(use), order="FIPS")
    print(f"  SVI tracts in {STATE}: {len(svi):,}")

    for col in use:
        if col != "FIPS":
            svi[col] = pd.to_numeric(svi[col], errors="coerce")
            n_sentinel = int((svi[col] == -999).sum())
            if n_sentinel:
                print(f"  {col}: nulling {n_sentinel} rows carrying the -999 sentinel")
            svi.loc[svi[col] == -999, col] = None
    svi["FIPS"] = safe_str(svi["FIPS"], width=11)

### CDC PLACES—the health measures that decide whether someone can leave

Four measures, chosen because each one describes a person who cannot simply walk out and drive
away:

- `MOBILITY`—difficulty walking or climbing stairs
- `SELFCARE`—difficulty dressing or bathing
- `INDEPLIVE`—difficulty doing errands alone
- `LACKTRPT`—lack of reliable transportation in the past twelve months

These are **model-based small-area estimates**, not counts of individuals. They tell you the
*prevalence* in a tract. Licence is `"Public Domain"`, stated by CDC in the dataset metadata.

In [ ]:
PLACES = "https://data.cdc.gov/resource/cwsq-ngmh.json"
MEASURES = ["MOBILITY", "SELFCARE", "INDEPLIVE", "LACKTRPT"]

with step("pull CDC PLACES"):
    inlist = ",".join(f"'{m_}'" for m_ in MEASURES)
    rows = get_json(PLACES, {
        "$select": "locationid,measureid,data_value,data_value_type,totalpopulation",
        "$where": f"stateabbr='{STATE}' AND measureid in ({inlist})",
        "$limit": 300000})
    places = pd.DataFrame(rows)
    print(f"  rows: {len(places):,}")
    if len(places) == 0:
        raise RuntimeError(f"CDC PLACES returned nothing for stateabbr='{STATE}'.")
    print(f"  value types present: {sorted(places['data_value_type'].unique())}")

    # Prefer crude prevalence where both are published; it is the one that describes the
    # population actually living there, which is what an evacuation planner cares about.
    types_present = set(places["data_value_type"])
    prefer = "Crude prevalence" if "Crude prevalence" in types_present else sorted(types_present)[0]
    print(f"  using: {prefer}")
    p = places[places["data_value_type"] == prefer].copy()
    p["data_value"] = pd.to_numeric(p["data_value"], errors="coerce")
    p["locationid"] = safe_str(p["locationid"], width=11)
    places_wide = p.pivot_table(index="locationid", columns="measureid",
                                values="data_value", aggfunc="first").reset_index()
    places_wide.columns = ["geo_id"] + [f"pct_{c.lower()}" for c in places_wide.columns[1:]]
    print(f"  tracts: {len(places_wide):,}   measures: {list(places_wide.columns[1:])}")

In [ ]:
with step("assemble and load vulnerability_tracts"):
    v = tracts.merge(svi.rename(columns={"FIPS": "geo_id"}), on="geo_id", how="left")
    n_after_svi = int(v["EP_DISABL"].notna().sum()) if "EP_DISABL" in v.columns else 0
    print(f"  tracts before SVI join: {len(tracts):,}   with SVI after: {n_after_svi:,}")
    v = v.merge(places_wide, on="geo_id", how="left")
    n_after_places = int(v.filter(like="pct_").notna().any(axis=1).sum())
    print(f"  with PLACES after join: {n_after_places:,}")

    v = v.rename(columns={"EP_DISABL": "pct_disability", "EP_AGE65": "pct_age_65_plus",
                          "EP_NOVEH": "pct_no_vehicle", "EP_LIMENG": "pct_limited_english",
                          "EP_MOBILE": "pct_mobile_homes", "EP_GROUPQ": "pct_group_quarters",
                          "EP_NOINT": "pct_no_internet", "E_TOTPOP": "svi_total_pop",
                          "RPL_THEME1": "svi_socioeconomic", "RPL_THEME2": "svi_household",
                          "RPL_THEME4": "svi_housing_transport"})
    n_tracts = load_df(v, "vulnerability_tracts")

## 9. Hazard, and the buildings full of people who cannot leave

**FEMA's National Risk Index** gives every census tract a hurricane and coastal-flooding exposure
figure, joined on an eleven-digit tract identifier. No spatial work, no geometry, just a join.

**One warning that matters.** The NRI's headline `RISK_SCORE` and every `*_RISKS` column are
computed as expected annual loss **multiplied by a social-vulnerability factor**. That means the
composite has social vulnerability baked into it, and using it as an input while *also* ranking on
vulnerability double-counts. We take the **expected-annual-loss and exposure** columns only, and
leave every composite alone. Same principle as the SVI composite above.

Licence: FEMA publishes this under a terms-of-use grant, not an open licence, and it requires
attribution—*"This product uses the Federal Emergency Management Agency's National Risk Index…
but is not endorsed by FEMA."* It also states the data are **"meant for planning purposes only"**,
which is exactly what you are doing.

In [ ]:
with step("pull FEMA National Risk Index"):   # NRI was defined in Section 8
    meta = get_json(NRI, {"f": "json"})
    available = {f["name"] for f in meta.get("fields", [])}
    wanted = ["TRACTFIPS", "POPULATION", "BUILDVALUE",
              "HRCN_AFREQ", "HRCN_EXPP", "HRCN_EALT", "HRCN_EALP",
              "CFLD_AFREQ", "CFLD_EXPP", "CFLD_EALT", "CFLD_EALP"]
    use = [f for f in wanted if f in available]
    print(f"  using {len(use)} fields; not present: {[f for f in wanted if f not in available]}")
    print(f"  deliberately NOT requested: RISK_SCORE, SOVI_SCORE, RESL_SCORE, CRF_VALUE "
          f"and every *_RISKS column")

    nri = arcgis_all(NRI, f"TRACTFIPS LIKE '{FIPS}%'", ",".join(use), order="TRACTFIPS")
    print(f"  NRI tracts in {STATE}: {len(nri):,}")
    for col in use:
        if col != "TRACTFIPS":
            nri[col] = pd.to_numeric(nri[col], errors="coerce")
            nri.loc[nri[col] == -9999, col] = None   # NRI's sentinel, same family as SVI's -999
    nri = nri.rename(columns={"TRACTFIPS": "geo_id"})
    nri["geo_id"] = safe_str(nri["geo_id"], width=11)
    n_hazard = load_df(nri, "hazard_tracts")

### Nursing homes

CMS publishes every Medicare-certified nursing home monthly, **with coordinates and certified bed
counts**. That combination is rarer than it sounds: the sibling CMS files for dialysis centres and
hospitals carry addresses only, so using those means geocoding—and Google Maps Platform forbids
storing geocoded coordinates beyond thirty days, which makes them a poor foundation for a table you
intend to keep. The Census Geocoder has no such restriction and is the right tool if you add them.

Note we resolve the download at runtime rather than hardcoding a URL. CMS embeds a content hash and
a month in its file paths, so any URL you copy today is broken by December.

In [ ]:
CMS_NH = "https://data.cms.gov/provider-data/api/1/datastore/query/4pq5-n9py/0"

with step("pull CMS nursing homes"):
    # CMS caps `limit` at 1000 on this endpoint. Ask for 2000 and it rejects the whole
    # request rather than clamping it. Read an API's limits before you trust them.
    rows, offset, PAGE = [], 0, 1000
    while offset < 40000:
        j = get_json(CMS_NH, {"limit": PAGE, "offset": offset})
        batch = j.get("results") or j.get("data") or []
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        if len(batch) < PAGE:
            break
    nh_all = pd.DataFrame(rows)
    print(f"  national nursing homes: {len(nh_all):,}  ({offset // PAGE + 1} requests)")
    if nh_all.empty:
        raise RuntimeError(
            "CMS returned no rows. Check that data.cms.gov is reachable from this runtime, and "
            "that dataset id 4pq5-n9py still resolves at "
            "https://data.cms.gov/provider-data/dataset/4pq5-n9py")

    # Never index a column you have not confirmed exists. CMS renames these between refreshes.
    def col(*candidates):
        lower = {c.lower(): c for c in nh_all.columns}
        for cand in candidates:
            if cand in lower:
                return lower[cand]
        return None

    state_col = col("state", "provider_state", "state_abbreviation")
    if state_col is None:
        raise RuntimeError(f"No state column in the CMS response. Columns: {list(nh_all.columns)}")

    nh = nh_all[nh_all[state_col].astype(str).str.strip().str.upper() == STATE].copy()
    print(f"  {STATE}: {len(nh):,} of {len(nh_all):,} national")
    if nh.empty:
        raise RuntimeError(
            f"No {STATE} nursing homes. Sample of what the state column contains: "
            f"{nh_all[state_col].dropna().unique()[:8].tolist()}")

    lat_c, lon_c = col("latitude"), col("longitude")
    bed_c = col("number_of_certified_beds", "certified_beds")
    wanted = [col("cms_certification_number_ccn", "federal_provider_number"),
              col("provider_name"), col("provider_address"), col("citytown", "city"),
              state_col, col("zip_code", "zip"), bed_c, col("ownership_type"), lat_c, lon_c]
    wanted = [c for c in wanted if c]
    nh = nh[wanted].copy()
    nh.columns = [c.lower() for c in nh.columns]
    lat_c = lat_c.lower() if lat_c else None
    bed_c = bed_c.lower() if bed_c else None

    for c_ in [x for x in (lat_c, lon_c.lower() if lon_c else None, bed_c) if x]:
        nh[c_] = pd.to_numeric(nh[c_], errors="coerce")

    # Report against what we actually got, not against what we hoped for.
    print(f"  with coordinates: {int(nh[lat_c].notna().sum()):,}" if lat_c
          else "  NO latitude column in this CMS release—you will need to geocode")
    print(f"  total certified beds in {STATE}: {int(nh[bed_c].fillna(0).sum()):,}" if bed_c
          else "  NO bed-count column in this CMS release")
    for c_ in nh.columns:
        if nh[c_].dtype == "object":
            nh[c_] = safe_str(nh[c_])
    n_care = load_df(nh, "care_facilities")

In [ ]:
with step("load emPOWER"):
    e = empower.copy()
    e.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c).lower() for c in e.columns]
    for col in e.columns:
        if e[col].dtype == "object":
            e[col] = safe_str(e[col])
    n_empower = load_df(e, "power_dependent_counties")

## 10. What the data actually says

This is the section to read even if you read nothing else, and it is where your story lane should
start. These are not our numbers. They are the country's.

In [ ]:
with step("payoff: the accessibility gap"):
    q = f"""
    SELECT
      COUNT(*)                                                   AS shelters,
      COUNTIF(wheelchair_accessible IS TRUE)                     AS wheelchair_yes,
      COUNTIF(wheelchair_accessible IS FALSE)                    AS wheelchair_no,
      COUNTIF(wheelchair_accessible IS NULL)                     AS wheelchair_unrecorded,
      COUNTIF(generator_onsite IS TRUE)                          AS generator_yes,
      COUNTIF(generator_onsite IS NULL)                          AS generator_unrecorded,
      COUNTIF(is_medical)                                        AS medical_needs,
      SUM(evacuation_capacity)                                   AS total_capacity
    FROM `{PROJECT}.{DATASET}.shelters`
    """
    r = list(bq.query(q).result())[0]
    pct = 100 * r.wheelchair_unrecorded / max(r.shelters, 1)
    print(f"  {STATE} shelters                        : {r.shelters:,}")
    print(f"  total evacuation capacity              : {int(r.total_capacity or 0):,}")
    print()
    print(f"  wheelchair access recorded as YES      : {r.wheelchair_yes:,}")
    print(f"  recorded as NO                         : {r.wheelchair_no:,}")
    print(f"  NOBODY RECORDED EITHER WAY             : {r.wheelchair_unrecorded:,}  ({pct:.0f}%)")
    print()
    print(f"  generator recorded as YES              : {r.generator_yes:,}")
    print(f"  generator not recorded                 : {r.generator_unrecorded:,}")
    print(f"  medical-needs designated               : {r.medical_needs:,} "
          f"({100*r.medical_needs/max(r.shelters,1):.1f}%)")

### Sit with that number for a second

Nationally, of 53,942 usable shelter records, 11,954 record wheelchair access as YES, 2,157 as NO,
and roughly **36,000 are blank**.

**For two-thirds of America's shelters, nobody has written down whether a person in a wheelchair
can get in.**

And Section 2 already showed you that Grounding with Google Maps cannot fill that gap either—five
different places, five different types, zero definite answers.

So the honest position for your agent, and one that will earn you more credit than pretending
otherwise: *"Here are the shelters near you. For eleven of them we know accessibility. For the
other sixty-four, nobody has recorded it, and that is a fact about the public record rather than
about the buildings."*

The second number is the scarcity. Nationally only **2%** of shelters carry a medical-needs
designation. Their median capacity is 200—identical to the median for all shelters, so they are
not bigger, just designated. There are vastly more people who cannot evacuate without medical
support than there are places equipped to receive them, and you can now measure that gap.

### The warning that comes with those numbers

Oregon reports 106 medical-needs shelters. Hawaii reports 88. Texas reports 31.

Oregon is not five times better prepared than Texas. **Oregon fills in the form.**

This file measures what states recorded, not what exists. Florida recording only 88
wheelchair-accessible shelters out of 1,533 is a reporting gap, not a statement about Florida's
buildings.

**Any team that ranks states, counties or neighbourhoods on these counts has measured bureaucracy
and called it risk.** That is the same trap as using a demographic proxy for a physical cause, and
a judge will ask you about it. The defensible move is to treat *unrecorded* as its own category—
visible, counted, and never silently folded into "no".

In [ ]:
with step("payoff: who is far from a usable shelter"):
    q = f"""
    WITH t AS (
      SELECT geo_id, lat, lon, total_pop, households, households_no_vehicle,
             pop_65_plus, pct_disability, pct_no_vehicle,
             SAFE_DIVIDE(households_no_vehicle, households) AS share_no_vehicle
      FROM `{PROJECT}.{DATASET}.vulnerability_tracts`
      WHERE lat IS NOT NULL AND total_pop > 0
    ),
    s AS (
      SELECT shelter_name, latitude, longitude, evacuation_capacity, wheelchair_accessible
      FROM `{PROJECT}.{DATASET}.shelters`
      WHERE latitude IS NOT NULL
    ),
    nearest AS (
      SELECT t.geo_id, t.total_pop, t.share_no_vehicle, t.pop_65_plus, t.pct_disability,
             MIN(ST_DISTANCE(ST_GEOGPOINT(t.lon, t.lat),
                             ST_GEOGPOINT(s.longitude, s.latitude))) / 1000 AS km_any,
             MIN(CASE WHEN s.wheelchair_accessible
                      THEN ST_DISTANCE(ST_GEOGPOINT(t.lon, t.lat),
                                       ST_GEOGPOINT(s.longitude, s.latitude)) END) / 1000
                                                                                AS km_wheelchair
      FROM t CROSS JOIN s
      GROUP BY 1, 2, 3, 4, 5
    )
    SELECT
      COUNT(*)                                        AS tracts,
      ROUND(APPROX_QUANTILES(km_any, 2)[OFFSET(1)], 1)        AS median_km_to_any_shelter,
      ROUND(APPROX_QUANTILES(km_wheelchair, 2)[OFFSET(1)], 1) AS median_km_to_wheelchair_shelter,
      COUNTIF(km_wheelchair IS NULL)                  AS tracts_with_no_wheelchair_shelter_at_all,
      ROUND(SUM(IF(km_any > 10, total_pop, 0)))       AS people_over_10km_from_any_shelter
    FROM nearest
    """
    display(bq.query(q).to_dataframe())
    print("\n  Two caveats, and say both out loud in your demo:")
    print("  1. 'km to a wheelchair-accessible shelter' counts only shelters where somebody")
    print("     recorded a YES. Given the coverage above, it is an upper bound on the good")
    print("     news, not a measurement of reality.")
    print("  2. Distances are computed on the tracts that have coordinates. See Section 8.")

### What that table said, in Florida

Median distance to **any** shelter: **1.9 km**. Median distance to one recorded as
**wheelchair accessible**: **7.1 km**.

If you need an accessible shelter, your nearest option is **nearly four times further away**.
And **493,461 Floridians** live more than ten kilometres from any shelter at all, which for a
household with no vehicle is not a distance, it is a wall.

That is your opening line. It is one number, it is defensible, and it does not require your
agent to work in order to be true.

Be careful with the honest version of it though: 7.1 km is the distance to the nearest shelter
*where somebody filled in the accessibility field*. The real nearest accessible shelter might
be closer, and nobody knows, and that is the point.

## 11. What we deliberately left out, and why

Every one of these was a decision, not an oversight. If a later session "fixes" one of them, this
is the record of why it was not broken.

| Left out | Why |
|---|---|
| `org_poc_name`, `org_poc_email`, `org_poc_phone`, `org_poc_after_hours_phone` | Named individuals. 47,006 rows nationally. Individual-level personal data is an automatic rejection at this event, and the only reliable defence is never requesting the columns |
| SVI Theme 3 and `RPL_THEMES` | Theme 3 is racial and ethnic minority status. Prohibited as a model input, required as a post-hoc audit. `RPL_THEMES` bakes it in with no column to drop |
| NRI `RISK_SCORE`, `*_RISKS`, `SOVI_SCORE`, `RESL_SCORE`, `CRF_VALUE` | Composites that multiply hazard by social vulnerability. Using them while also ranking on vulnerability double-counts |
| `ada_compliant` | 63 YES rows nationally. Empty, and keeping it invites somebody to filter on it |
| `in_500_yr_floodplain`, `pre_landfall_shelter` | Zero and fourteen YES rows nationally. Same reasoning |
| Google geocoding for the address-only CMS files | Maps Platform forbids storing geocoded coordinates beyond thirty days, so they cannot be the basis of a table you keep. Use the Census Geocoder |
| Real-time storm feeds | NHC's live GIS products only exist while a watch or warning is active. In October they are empty. This is a planning tool |
| Elderly-living-alone (ACS B11007) | Needs a Census API key, obtained before the event. Bonus rather than core—an avoidable single point of failure for 150 people |

**Race is not in your model. It belongs in your audit.** When your agent produces a ranked list of
neighbourhoods, join it back to the demographics afterwards and ask whether the recommendation
lands disproportionately on any group. If it does, that is a finding to report, not a bug to hide—
and reporting it will earn you more credit than a clean-looking model that nobody checked.

## 12. Validate before you build

Every check below queries the **BigQuery tables you just loaded**, not the dataframes in memory.
That distinction has bitten this project before: a validation suite that reads the wrong store is
worse than no validation suite, because it is reassuring.

The framing worth remembering: **the errors that hurt you are the ones that don't raise.**

In [ ]:
with step("validation"):
    CHECKS.clear()
    T = f"`{PROJECT}.{DATASET}"

    r = list(bq.query(f"SELECT COUNT(*) n FROM {T}.shelters`").result())[0]
    check("shelters table is not empty", r.n > 0, f"{r.n:,} rows")

    cols = [c.column_name for c in bq.query(f"""
        SELECT column_name FROM {T}.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name = 'shelters'""").result()]
    check("no personal-data columns reached BigQuery",
          not any(c.startswith("org_poc") for c in cols), f"{len(cols)} columns")

    # Per-state bounding box, never a national one. A national check passes while an entire
    # state sits in the Indian Ocean.
    r = list(bq.query(f"""
        SELECT COUNT(*) n, COUNTIF(latitude BETWEEN 17 AND 72) ok,
               ROUND(MIN(latitude),2) lo, ROUND(MAX(latitude),2) hi
        FROM {T}.shelters` WHERE latitude IS NOT NULL""").result())[0]
    check("every shelter latitude is a latitude", r.ok == r.n, f"range {r.lo} to {r.hi}")

    # FEMA's own fips_code is empty, so we derived county from the coordinates instead.
    # Majority, not equality: a point on a shoreline can legitimately miss every polygon.
    if "county_fips" in cols:
        r = list(bq.query(f"""
            SELECT COUNT(*) n, COUNTIF(county_fips IS NOT NULL) matched,
                   COUNTIF(SUBSTR(county_fips,1,2) = '{FIPS}') in_state,
                   COUNT(DISTINCT county_fips) counties
            FROM {T}.shelters`""").result())[0]
        check("shelters were placed in a county from their coordinates",
              r.matched >= 0.9 * r.n, f"{r.matched:,} of {r.n:,} across {r.counties} counties")
        check("those counties are in this state", r.in_state == r.matched,
              f"{r.in_state:,} of {r.matched:,}")
    else:
        check("shelters were placed in a county from their coordinates", False,
              "county_fips column absent—the spatial join did not run")

    r = list(bq.query(f"""
        SELECT COUNTIF(evacuation_capacity < 0) neg,
               COUNTIF(evacuation_capacity > 100000) huge, COUNT(*) n
        FROM {T}.shelters`""").result())[0]
    check("capacities are physically possible", r.neg == 0 and r.huge == 0,
          f"{r.neg} negative, {r.huge} above 100k")

    r = list(bq.query(f"""
        SELECT COUNTIF(wheelchair_accessible IS TRUE) y,
               COUNTIF(wheelchair_accessible IS FALSE) n2,
               COUNTIF(wheelchair_accessible IS NULL) u FROM {T}.shelters`""").result())[0]
    check("wheelchair flag parsed to a real boolean", (r.y + r.n2) > 0,
          f"{r.y} true / {r.n2} false / {r.u} unrecorded")
    check("unrecorded kept separate from 'no'", r.u > 0,
          "if this is 0 in your state, say so on your slide—it would be unusual")

    r = list(bq.query(f"SELECT COUNT(*) n, COUNTIF(total_pop IS NOT NULL) acs, "
                      f"COUNTIF(lat IS NOT NULL) coords, COUNTIF(pct_disability IS NOT NULL) svi "
                      f"FROM {T}.vulnerability_tracts`").result())[0]
    check("vulnerability_tracts is not empty", r.n > 0, f"{r.n:,} tracts")
    check("every tract carries its ACS demographics", r.acs >= 0.99 * r.n,
          f"{r.acs:,} of {r.n:,}—ACS is the spine, so this should be ~100%")
    check("SVI joined to most tracts", r.svi >= 0.9 * r.n,
          f"{r.svi:,} of {r.n:,} ({100*r.svi/max(r.n,1):.0f}%)—both are 2020 vintage")
    # Not a failure, a disclosure: coordinates come from a 2010-vintage geometry table.
    # Threshold is 0.95 because NRI centroids are 2020 vintage and should cover nearly
    # everything. A drop to the mid-60s means the centroid request silently failed and we
    # fell back to the 2010 geometry table.
    check("tracts have coordinates", r.coords >= 0.95 * r.n,
          f"{r.coords:,} of {r.n:,} ({100*r.coords/max(r.n,1):.0f}%)—from NRI centroids, "
          f"with the 2010 geometry table as fallback")

    r = list(bq.query(f"SELECT COUNT(*) n, COUNTIF(LENGTH(geo_id)=11) ok "
                      f"FROM {T}.vulnerability_tracts`").result())[0]
    check("every tract id is 11 characters", r.ok == r.n, f"{r.ok:,} of {r.n:,}")

    # Two counts around the join, and fail on the ratio rather than the absolute.
    r = list(bq.query(f"""
        SELECT (SELECT COUNT(*) FROM {T}.vulnerability_tracts`) before,
               (SELECT COUNT(*) FROM {T}.vulnerability_tracts` v
                JOIN {T}.hazard_tracts` h USING (geo_id)) after""").result())[0]
    # Threshold raised from 0.5 now that both sides are 2020 vintage. A drop below 90% would
    # mean something new is wrong, rather than the drift we already know about.
    check("hazard tracts join to vulnerability tracts", r.after >= 0.9 * r.before,
          f"{r.before:,} tracts before, {r.after:,} after "
          f"({100*r.after/max(r.before,1):.0f}%)")

    # The join the county work exists for: can a shelter reach its county's emPOWER row?
    if "county_fips" in cols:
        r = list(bq.query(f"""
            SELECT (SELECT COUNT(*) FROM {T}.shelters` WHERE county_fips IS NOT NULL) before,
                   (SELECT COUNT(*) FROM {T}.shelters` s
                    JOIN {T}.power_dependent_counties` e ON e.fips_code = s.county_fips) after
            """).result())[0]
        check("shelters join to their county's emPOWER row", r.after >= 0.9 * r.before,
              f"{r.before:,} shelters with a county, {r.after:,} matched to emPOWER")

    r = list(bq.query(f"""
        SELECT COUNTIF(lat NOT BETWEEN 17 AND 72) bad_lat,
               COUNTIF(lon NOT BETWEEN -180 AND -60) bad_lon
        FROM {T}.vulnerability_tracts` WHERE lat IS NOT NULL""").result())[0]
    check("tract coordinates are degrees, not Web Mercator metres",
          r.bad_lat == 0 and r.bad_lon == 0,
          f"{r.bad_lat} bad latitudes, {r.bad_lon} bad longitudes")

    r = list(bq.query(f"""
        SELECT COUNTIF(pct_disability = -999) a, COUNTIF(pct_no_vehicle = -999) b
        FROM {T}.vulnerability_tracts`""").result())[0]
    check("no -999 sentinels survived", (r.a or 0) + (r.b or 0) == 0,
          f"{(r.a or 0) + (r.b or 0)} found")

    r = list(bq.query(f"SELECT COUNT(*) n, COUNTIF(latitude IS NOT NULL) c "
                      f"FROM {T}.care_facilities`").result())[0]
    check("care facilities loaded with coordinates", r.n > 0 and r.c > 0,
          f"{r.n:,} facilities, {r.c:,} with coordinates")

    r = list(bq.query(f"SELECT COUNT(*) n FROM {T}.power_dependent_counties`").result())[0]
    check("emPOWER counties loaded", r.n > 0, f"{r.n:,} counties")

    r = list(bq.query(f"SELECT COUNTIF(is_medical) n FROM {T}.shelters`").result())[0]
    check("this state has at least one medical-needs shelter", r.n > 0,
          f"{r.n}—if this is low, read Section 4 again before building your demo on it")

    df_checks = pd.DataFrame(CHECKS)
    print()
    display(df_checks)
    failed = int((df_checks["result"] == "FAIL").sum())
    print(f"\n{len(df_checks) - failed} passed, {failed} failed")

## 13. Framing your agent—read this before you write any code

This notebook stops here on purpose. The agent is yours, and the design decisions in it are what
you are judged on. But four things about *this* challenge will cost you time if you meet them by
accident, and one of them is architectural.

### The model generation you pick changes your architecture

This one is worth knowing before you design anything, because it changed between model
generations and most of what you will read online describes the old behaviour.

**On Gemini 2.5 and older**, a built-in tool could not share an agent with a function tool of your
own. The request was rejected outright:

```
400 INVALID_ARGUMENT—"Unable to submit request because Multiple tools are supported
only when they are all search tools."
```

**On Gemini 3.x it works.** We put `google_maps_grounding` and a BigQuery function tool in a single
ADK agent, ran it, and watched both fire in one turn—the agent queried our shelters table *and*
grounded against Google Maps, and said so separately in its answer. Verified 2026-08-09 on
`gemini-3.6-flash`; the identical request on `gemini-2.5-flash` still returns the 400.

So: **use a 3.x model and you can keep one agent.** If you pin an older one, you need this instead,
and it is a perfectly good architecture either way:

- a **maps agent** holding `google_maps_grounding` and nothing else
- your **root agent** holding your own tools, an MCP server, and the maps agent wrapped in
  `AgentTool`

`GoogleMapsGroundingTool` still takes no constructor arguments and still does not accept
`bypass_multi_tools_limit`, so on an older model the sub-agent is the only route.

**One warning you will see and should ignore:**

```
Tools at indices [0] are not compatible with automatic function calling (AFC). AFC is disabled.
```

That is the client telling you it will not run the function-calling loop for you. ADK runs its own
loop, so everything works. It appears once per turn and it is noise.

### What Grounding with Google Maps will and will not do

| Will | Will not |
|---|---|
| Find places, addresses, ratings, hours | Give you a driving time or a distance |
| Tell you if somewhere is open **right now** | Give you a route or a polyline |
| Tell you a business has permanently closed, or renamed | Answer anything but English |
| Say honestly when it does not know | Answer an *area* question about accessibility |

**On accessibility, be precise, because the shape of your question decides the answer.** Ask it
*"which shelters in this county take a wheelchair"* and it cannot help—Section 2 showed that.
Ask it about **one named building at one address** and in our testing it answered 5 times out of 8,
and agreed with FEMA's record 4 times out of 5 where both had a view.

That is a design direction, not a dead end, and it is one of the more valuable things you could
build this afternoon.

Routing and Search Along Route are Private Preview. **Do not design a deliverable around a route.**
Expect five to eight seconds per grounded call—that is normal, not a bug.

### Attribution is a requirement, not a nicety

If you display grounded output, you must display its Google Maps sources **immediately following
the content they support**, viewable within one user interaction. The string "Google Maps" must not
be restyled, re-capitalised, wrapped across lines or translated—set `translate="no"`.

The response gives you what you need: `groundingChunks[].maps` carries `place_id`, `title` and
`uri`, and `groundingSupports` maps each sentence back to its sources. Two practical notes:
`segment.start_index` is `None` on the first segment—treat it as 0—and `confidence_scores` is
never populated, so do not build a UI that depends on it.

### What you may keep

You may **not** cache, store or export Google Maps content. You **may** keep `place_id` and
`review_id`—those are explicitly exempt.

So the shape that is both legal and correct is the same shape: **our data is the memory, Maps is
the live lookup, and the only thing you carry between them is the pointer.** Store a `place_id`
against your shelter row, re-ground at runtime, and you have a system that is compliant and current
at the same time.

### Where the good version separates from the merely working one

- It **counts what is unrecorded** rather than treating blank as no.
- It says which claims come from a 2022 federal record and which were verified live seconds ago.
- It audits its own output against the demographics it was forbidden to use as an input.
- It knows that a shelter twelve kilometres away is not a destination for someone with no vehicle.
- It is honest when it does not know, in a domain where a confident wrong answer is the worst
  possible output.

## 14. Appendix—the diagnostic block

One cell, one paste. If you need help from a coach, run this and paste the output. It beats twenty
screenshots, and it is how we test this notebook between events.

In [ ]:
print("=" * 74)
print(f"C4 EVACUATION READINESS—DIAGNOSTIC  |  state={STATE}  project={PROJECT}")
print("=" * 74)

summary = {}
for t in ["shelters", "vulnerability_tracts", "hazard_tracts",
          "care_facilities", "power_dependent_counties"]:
    try:
        summary[t] = bq.get_table(f"{PROJECT}.{DATASET}.{t}").num_rows
    except Exception as e:
        summary[t] = f"MISSING ({type(e).__name__})"

r = list(bq.query(f"""
  SELECT COUNTIF(wheelchair_accessible IS TRUE) wc_yes,
         COUNTIF(wheelchair_accessible IS FALSE) wc_no,
         COUNTIF(wheelchair_accessible IS NULL) wc_unrecorded,
         COUNTIF(is_medical) medical,
         SUM(evacuation_capacity) capacity
  FROM `{PROJECT}.{DATASET}.shelters`""").result())[0]

failed = [c for c in CHECKS if c["result"] == "FAIL"]
print(json.dumps({
    "state": STATE, "state_fips": FIPS, "acs_table": ACS_TABLE,
    "tables": summary,
    "shelters": {"wheelchair_yes": r.wc_yes, "wheelchair_no": r.wc_no,
                 "wheelchair_unrecorded": r.wc_unrecorded,
                 "medical_needs": r.medical,
                 "total_capacity": int(r.capacity or 0)},
    "checks": {"total": len(CHECKS), "passed": len(CHECKS) - len(failed),
               "failed": [c["check"] for c in failed]},
    "step_seconds": STEPS,
    # Test for None, not for truth. A step that took 0.0s ran; a step that is missing did not.
    "total_seconds": round(sum(v for v in STEPS.values() if v is not None), 1),
}, indent=1, default=str))
print("=" * 74)